<a href="https://colab.research.google.com/github/saeyeon055-hue/Bio-AI-Learning-Path/blob/main/ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. XOR by PyTorch

In [ ]:
import numpy as np
import torch
import torch.nn as nn # neural net을 디자인하는 모듈
from sklearn.metrics import accuracy_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 데이터 읽기 함수
def load_dataset(file,device): # 2개의 파라미터(file: 읽어들일 파일명;file name,device: cuda 혹은 cpu; cuda는 GPU라는 프로세스 유닛에서 실행해라. cpu는 cpu에서 실행해라 라는 의미)
  data=np.loadtxt(file) # loadtxt: 데이터 읽어들이는 함수
  print('DATA=',data)

  input_features = data[:, 0:-1] # ':' 행에 해당하는 것은 다 읽어라, '0:-1' 열은 맨 뒤에거 빼고 읽어
  print('INPUT_FEATURES=',input_features)

  labels = np.reshape(data[:,-1],(4,1)) # np.reshape: 행과 열을 바꾸는 함수 reshape(data, 행렬 정보)
  print('LABELS=', labels)

  input_features = torch.tensor(input_features, dtype=torch.float).to(device) # PyTorch는 numpy array를 사용하지 않고 tensor의 자료 구조를 사용하기 때문에 torch.tensor로 변환해줘야한다.
  labels = torch.tensor(labels, dtype=torch.float).to(device)

  return (input_features, labels)

# 모델 평가 결과 계산을 위해 텐서를 리스트로 변환하는 함수
def tensor2list(input_tensor):
  return input_tensor.cpu().detach().numpy().tolist()

In [ ]:
# GPU 사용 가능 여부 확인(torch.cuda.is_available)
if torch.cuda.is_available():
  device='cuda'
else:
  device='cpu'

input_features, labels = load_dataset('/content/drive/MyDrive/Colab Notebooks/기계학습/train.txt', device)

## NN 모델 만들기
model = nn.Sequential(
    nn.Linear(2, 2, bias=True), nn.Sigmoid(),
    nn.Linear(2, 1, bias=True), nn.Sigmoid().to(device)
)

# 이진분류 크로스에늩로피 비용 함수
loss_func = torch.nn.BCELoss().to(device)

# 옵티마이저 함수 (역전파 알고리즘을 수행할 함수)
optimizer = torch.optim.SGD(model.parameters(), lr=1)

# 학습 모드 셋팅
model.train()

DATA= [[0. 0. 0.]
 [0. 1. 1.]
 [1. 0. 1.]
 [1. 1. 0.]]
INPUT_FEATURES= [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
LABELS= [[0.]
 [1.]
 [1.]
 [0.]]


Sequential(
  (0): Linear(in_features=2, out_features=2, bias=True)
  (1): Sigmoid()
  (2): Linear(in_features=2, out_features=1, bias=True)
  (3): Sigmoid()
)

In [ ]:
# 모델 학습
for epoch in range(1001):

  # 기울기 계산한 것들 초기화
  optimizer.zero_grad()

  # H(x) 계산: forward 연산
  hypothesis =model(input_features)

  # 비용 계산
  cost = loss_func(hypothesis, labels)

  # 역전파 수행
  cost.backward()
  optimizer.step()

  # 100 에폭마다 비용 출력
  if epoch % 100 == 0:
    print(epoch, cost.item())

0 0.6565458178520203
100 0.528904914855957
200 0.30812257528305054
300 0.11544300615787506
400 0.05984964966773987
500 0.03893052414059639
600 0.028485748916864395
700 0.02233029529452324
800 0.018304504454135895
900 0.015478736720979214
1000 0.013391830027103424


In [ ]:
# 평가 모드 셋팅 (학습 시 적용했던 드랍 아웃 여부 등을 비적용)
model.eval()

# 역전파를 적용하지 않도록 context manager 설정
with torch.no_grad():
  hypothesis = model(input_features)
  logits = (hypothesis > 0.5).float()
  predicts = tensor2list(logits)
  golds = tensor2list(labels)
  print('PRED=', predicts)
  print('GOLD]', golds)
  print('Accuracy : {0:f}'.format(accuracy_score(golds, predicts)))

PRED= [[0.0], [1.0], [1.0], [0.0]]
GOLD] [[0.0], [1.0], [1.0], [0.0]]
Accuracy : 1.000000


# 2. Wide ANN
> Hidden layer를 2 * 2에서 2 * 10으로 변경

> Widening은 선의 개수를 늘리는 효과

In [ ]:
if torch.cuda.is_available():
  device='cuda'
else:
  device='cpu'

input_features, labels = load_dataset('/content/drive/MyDrive/Colab Notebooks/기계학습/train.txt', device)

## NN 모델 만들기
model = nn.Sequential(
    nn.Linear(2, 10, bias=True), nn.Sigmoid(),
    nn.Linear(10, 1, bias=True), nn.Sigmoid().to(device)
)

# 이진분류 크로스엔트로피 비용 함수
loss_func = torch.nn.BCELoss().to(device)

# 옵티마이저 함수 (역전파 알고리즘을 수행할 함수)
optimizer = torch.optim.SGD(model.parameters(), lr=1)

# 학습 모드 셋팅
model.train()

DATA= [[0. 0. 0.]
 [0. 1. 1.]
 [1. 0. 1.]
 [1. 1. 0.]]
INPUT_FEATURES= [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
LABELS= [[0.]
 [1.]
 [1.]
 [0.]]


Sequential(
  (0): Linear(in_features=2, out_features=10, bias=True)
  (1): Sigmoid()
  (2): Linear(in_features=10, out_features=1, bias=True)
  (3): Sigmoid()
)

In [ ]:
# 모델 학습
for epoch in range(1001):

  # 기울기 계산한 것들 초기화
  optimizer.zero_grad()

  # H(x) 계산: forward 연산
  hypothesis =model(input_features)

  # 비용 계산
  cost = loss_func(hypothesis, labels)

  # 역전파 수행
  cost.backward()
  optimizer.step()

  # 100 에폭마다 비용 출력
  if epoch % 100 == 0:
    print(epoch, cost.item())

0 0.6932336091995239
100 0.6918482780456543
200 0.6853496432304382
300 0.6191054582595825
400 0.3300333023071289
500 0.0975780040025711
600 0.046307794749736786
700 0.029021821916103363
800 0.02079770341515541
900 0.01608039252460003
1000 0.01304871216416359


In [ ]:
# 평가 모드 셋팅 (학습 시 적용했던 드랍 아웃 여부 등을 비적용)
model.eval()

# 역전파를 적용하지 않도록 context manager 설정
with torch.no_grad():
  hypothesis = model(input_features)
  logits = (hypothesis > 0.5).float()
  predicts = tensor2list(logits)
  golds = tensor2list(labels)
  print('PRED=', predicts)
  print('GOLD]', golds)
  print('Accuracy : {0:f}'.format(accuracy_score(golds, predicts)))

PRED= [[0.0], [1.0], [1.0], [0.0]]
GOLD] [[0.0], [1.0], [1.0], [0.0]]
Accuracy : 1.000000


# 3. Shallow ANN
> Hidden layer를 없애고 single-layer perceptron으로 변경

In [ ]:
if torch.cuda.is_available():
  device='cuda'
else:
  device='cpu'

input_features, labels = load_dataset('/content/drive/MyDrive/Colab Notebooks/기계학습/train.txt', device)

## NN 모델 만들기
model = nn.Sequential(
    nn.Linear(2, 1, bias=True), nn.Sigmoid().to(device)
)

# 이진분류 크로스엔트로피 비용 함수
loss_func = torch.nn.BCELoss().to(device)

# 옵티마이저 함수 (역전파 알고리즘을 수행할 함수)
optimizer = torch.optim.SGD(model.parameters(), lr=1)

# 학습 모드 셋팅
model.train()

DATA= [[0. 0. 0.]
 [0. 1. 1.]
 [1. 0. 1.]
 [1. 1. 0.]]
INPUT_FEATURES= [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
LABELS= [[0.]
 [1.]
 [1.]
 [0.]]


Sequential(
  (0): Linear(in_features=2, out_features=1, bias=True)
  (1): Sigmoid()
)

In [ ]:
# 모델 학습
for epoch in range(10001):

  # 기울기 계산한 것들 초기화
  optimizer.zero_grad()

  # H(x) 계산: forward 연산
  hypothesis =model(input_features)

  # 비용 계산
  cost = loss_func(hypothesis, labels)

  # 역전파 수행
  cost.backward()
  optimizer.step()

  # 1000 에폭마다 비용 출력
  if epoch % 1000 == 0:
    print(epoch, cost.item())

0 0.6931471824645996
1000 0.6931471824645996
2000 0.6931471824645996
3000 0.6931471824645996
4000 0.6931471824645996
5000 0.6931471824645996
6000 0.6931471824645996
7000 0.6931471824645996
8000 0.6931471824645996
9000 0.6931471824645996
10000 0.6931471824645996


In [ ]:
# 평가 모드 셋팅 (학습 시 적용했던 드랍 아웃 여부 등을 비적용)
model.eval()

# 역전파를 적용하지 않도록 context manager 설정
with torch.no_grad():
  hypothesis = model(input_features)
  logits = (hypothesis > 0.5).float()
  predicts = tensor2list(logits)
  golds = tensor2list(labels)
  print('PRED=', predicts)
  print('GOLD]', golds)
  print('Accuracy : {0:f}'.format(accuracy_score(golds, predicts)))

PRED= [[0.0], [0.0], [0.0], [0.0]]
GOLD] [[0.0], [1.0], [1.0], [0.0]]
Accuracy : 0.500000


> 학습 속도는 빠르지만 10,000 epoch를 수행해도 문제를 풀지 못함.

- 왜냐하면 single-layer perceptron은 linear separable problem 만 해결 가능하기 때문.

# 4. Deep ANN
> Hidden layer 층을 1개에서 2개로 변경

In [ ]:
if torch.cuda.is_available():
  device='cuda'
else:
  device='cpu'

input_features, labels = load_dataset('/content/drive/MyDrive/Colab Notebooks/기계학습/train.txt', device)

## NN 모델 만들기
model = nn.Sequential(
    nn.Linear(2, 2, bias=True), nn.Sigmoid(),
    nn.Linear(2, 2, bias=True), nn.Sigmoid(),
    nn.Linear(2, 1, bias=True), nn.Sigmoid()).to(device)

# 이진분류 크로스엔트피 비용 함수
loss_func = torch.nn.BCELoss().to(device)

# 옵티마이저 함수 (역전파 알고리즘을 수행할 함수)
optimizer = torch.optim.SGD(model.parameters(), lr=1)

# 학습 모드 셋팅
model.train()

DATA= [[0. 0. 0.]
 [0. 1. 1.]
 [1. 0. 1.]
 [1. 1. 0.]]
INPUT_FEATURES= [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
LABELS= [[0.]
 [1.]
 [1.]
 [0.]]


Sequential(
  (0): Linear(in_features=2, out_features=2, bias=True)
  (1): Sigmoid()
  (2): Linear(in_features=2, out_features=2, bias=True)
  (3): Sigmoid()
  (4): Linear(in_features=2, out_features=1, bias=True)
  (5): Sigmoid()
)

In [ ]:
# 모델 학습
for epoch in range(4001):

  # 기울기 계산한 것들 초기화
  optimizer.zero_grad()

  # H(x) 계산: forward 연산
  hypothesis =model(input_features)

  # 비용 계산
  cost = loss_func(hypothesis, labels)

  # 역전파 수행
  cost.backward()
  optimizer.step()

  # 400 에폭마다 비용 출력
  if epoch % 400 == 0:
    print(epoch, cost.item())

0 0.6311635971069336
400 0.490317702293396
800 0.010756785050034523
1200 0.004827172961086035
1600 0.0030831736512482166
2000 0.0022559459321200848
2400 0.0017749276012182236
2800 0.0014610359212383628
3200 0.00124040013179183
3600 0.0010770109947770834
4000 0.0009511990356259048


In [ ]:
# 평가 모드 셋팅 (학습 시 적용했던 드랍 아웃 여부 등을 비적용)
model.eval()

# 역전파를 적용하지 않도록 context manager 설정
with torch.no_grad():
  hypothesis = model(input_features)
  logits = (hypothesis > 0.5).float()
  predicts = tensor2list(logits)
  golds = tensor2list(labels)
  print('PRED=', predicts)
  print('GOLD]', golds)
  print('Accuracy : {0:f}'.format(accuracy_score(golds, predicts)))

PRED= [[0.0], [1.0], [1.0], [0.0]]
GOLD] [[0.0], [1.0], [1.0], [0.0]]
Accuracy : 1.000000


> deeping은 선을 구부리는 효과

## 4-1. Hidden layer 층을 1개에서 7개로 변경

In [ ]:
if torch.cuda.is_available():
  device='cuda'
else:
  device='cpu'

input_features, labels = load_dataset('/content/drive/MyDrive/Colab Notebooks/기계학습/train.txt', device)

## NN 모델 만들기
model = nn.Sequential(
    nn.Linear(2, 10, bias=True), nn.Sigmoid(),
    nn.Linear(10, 10, bias=True), nn.Sigmoid(),
    nn.Linear(10, 10, bias=True), nn.Sigmoid(),
    nn.Linear(10, 10, bias=True), nn.Sigmoid(),
    nn.Linear(10, 10, bias=True), nn.Sigmoid(),
    nn.Linear(10, 10, bias=True), nn.Sigmoid(),
    nn.Linear(10, 10, bias=True), nn.Sigmoid(),
    nn.Linear(10, 10, bias=True), nn.Sigmoid(),
    nn.Linear(10, 1, bias=True), nn.Sigmoid()).to(device)

# 이진분류 크로스엔트피 비용 함수
loss_func = torch.nn.BCELoss().to(device)

# 옵티마이저 함수 (역전파 알고리즘을 수행할 함수)
optimizer = torch.optim.SGD(model.parameters(), lr=1)

# 학습 모드 셋팅
model.train()

DATA= [[0. 0. 0.]
 [0. 1. 1.]
 [1. 0. 1.]
 [1. 1. 0.]]
INPUT_FEATURES= [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
LABELS= [[0.]
 [1.]
 [1.]
 [0.]]


Sequential(
  (0): Linear(in_features=2, out_features=10, bias=True)
  (1): Sigmoid()
  (2): Linear(in_features=10, out_features=10, bias=True)
  (3): Sigmoid()
  (4): Linear(in_features=10, out_features=10, bias=True)
  (5): Sigmoid()
  (6): Linear(in_features=10, out_features=10, bias=True)
  (7): Sigmoid()
  (8): Linear(in_features=10, out_features=10, bias=True)
  (9): Sigmoid()
  (10): Linear(in_features=10, out_features=10, bias=True)
  (11): Sigmoid()
  (12): Linear(in_features=10, out_features=10, bias=True)
  (13): Sigmoid()
  (14): Linear(in_features=10, out_features=10, bias=True)
  (15): Sigmoid()
  (16): Linear(in_features=10, out_features=1, bias=True)
  (17): Sigmoid()
)

In [ ]:
# 모델 학습
for epoch in range(10001):

  # 기울기 계산한 것들 초기화
  optimizer.zero_grad()

  # H(x) 계산: forward 연산
  hypothesis =model(input_features)

  # 비용 계산
  cost = loss_func(hypothesis, labels)

  # 역전파 수행
  cost.backward()
  optimizer.step()

  # 1000 에폭마다 비용 출력
  if epoch % 1000 == 0:
    print(epoch, cost.item())

0 0.706458568572998
1000 0.6931471824645996
2000 0.6931471824645996
3000 0.6931471824645996
4000 0.6931471824645996
5000 0.6931471824645996
6000 0.6931471824645996
7000 0.6931471824645996
8000 0.6931471824645996
9000 0.6931471824645996
10000 0.6931471824645996


In [ ]:
# 평가 모드 셋팅 (학습 시 적용했던 드랍 아웃 여부 등을 비적용)
model.eval()

# 역전파를 적용하지 않도록 context manager 설정
with torch.no_grad():
  hypothesis = model(input_features)
  logits = (hypothesis > 0.5).float()
  predicts = tensor2list(logits)
  golds = tensor2list(labels)
  print('PRED=', predicts)
  print('GOLD]', golds)
  print('Accuracy : {0:f}'.format(accuracy_score(golds, predicts)))

PRED= [[1.0], [0.0], [0.0], [0.0]]
GOLD] [[0.0], [1.0], [1.0], [0.0]]
Accuracy : 0.250000


> 10,000 epoch를 돌려도 학습이 안됨
- 2차 winter season 원인

- 문제 해결을 위해 더 복잡한 층을 설계했는데 문제 해결이 되지 않음

> 원인: Vanishing Gradient
- 초기 hidden layer에서 아래로 잘 전파되다가, 깊어지니까 희미해지면서 전파가 되지 않게 됨.

- 결국 아래층에는 업데이트가 되지 않음.

# 5. Sigmoid to ReLU

In [ ]:
if torch.cuda.is_available():
  device='cuda'
else:
  device='cpu'

input_features, labels = load_dataset('/content/drive/MyDrive/Colab Notebooks/기계학습/train.txt', device)

## NN 모델 만들기 (마지막 layer는 0과 1 사이 값을 출력하도록 하기 위해서 signmoid 유지)
model = nn.Sequential(
    nn.Linear(2, 10, bias=True), nn.ReLU(),
    nn.Linear(10, 10, bias=True), nn.ReLU(),
    nn.Linear(10, 10, bias=True), nn.ReLU(),
    nn.Linear(10, 10, bias=True), nn.ReLU(),
    nn.Linear(10, 10, bias=True), nn.ReLU(),
    nn.Linear(10, 10, bias=True), nn.ReLU(),
    nn.Linear(10, 10, bias=True), nn.ReLU(),
    nn.Linear(10, 10, bias=True), nn.ReLU(),
    nn.Linear(10, 1, bias=True), nn.Sigmoid()).to(device)

# 이진분류 크로스엔트피 비용 함수
loss_func = torch.nn.BCELoss().to(device)

# 옵티마이저 함수 (역전파 알고리즘을 수행할 함수)
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)

# 학습 모드 셋팅
model.train()

DATA= [[0. 0. 0.]
 [0. 1. 1.]
 [1. 0. 1.]
 [1. 1. 0.]]
INPUT_FEATURES= [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
LABELS= [[0.]
 [1.]
 [1.]
 [0.]]


Sequential(
  (0): Linear(in_features=2, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=10, bias=True)
  (3): ReLU()
  (4): Linear(in_features=10, out_features=10, bias=True)
  (5): ReLU()
  (6): Linear(in_features=10, out_features=10, bias=True)
  (7): ReLU()
  (8): Linear(in_features=10, out_features=10, bias=True)
  (9): ReLU()
  (10): Linear(in_features=10, out_features=10, bias=True)
  (11): ReLU()
  (12): Linear(in_features=10, out_features=10, bias=True)
  (13): ReLU()
  (14): Linear(in_features=10, out_features=10, bias=True)
  (15): ReLU()
  (16): Linear(in_features=10, out_features=1, bias=True)
  (17): Sigmoid()
)

In [ ]:
# 모델 학습
for epoch in range(3001):

  # 기울기 계산한 것들 초기화
  optimizer.zero_grad()

  # H(x) 계산: forward 연산
  hypothesis =model(input_features)

  # 비용 계산
  cost = loss_func(hypothesis, labels)

  # 역전파 수행
  cost.backward()
  optimizer.step()

  # 300 에폭마다 비용 출력
  if epoch % 300 == 0:
    print(epoch, cost.item())

0 0.6965017318725586
300 0.6930959224700928
600 0.69305819272995
900 0.6929864287376404
1200 0.6927822232246399
1500 0.6913377046585083
1800 0.003306519938632846
2100 0.0002116301329806447
2400 9.490660886513069e-05
2700 5.842854443471879e-05
3000 4.135034396313131e-05


In [ ]:
# 평가 모드 셋팅 (학습 시 적용했던 드랍 아웃 여부 등을 비적용)
model.eval()

# 역전파를 적용하지 않도록 context manager 설정
with torch.no_grad():
  hypothesis = model(input_features)
  logits = (hypothesis > 0.5).float()
  predicts = tensor2list(logits)
  golds = tensor2list(labels)
  print('PRED=', predicts)
  print('GOLD]', golds)
  print('Accuracy : {0:f}'.format(accuracy_score(golds, predicts)))

PRED= [[0.0], [1.0], [1.0], [0.0]]
GOLD] [[0.0], [1.0], [1.0], [0.0]]
Accuracy : 1.000000


# 6. Dropout by PyTorch
> 학습 과정 중에 지정된 비율로 임의의 연결을 끊음으로써 일반화 성능을 개선하는 방법

In [ ]:
if torch.cuda.is_available():
  device='cuda'
else:
  device='cpu'

input_features, labels = load_dataset('/content/drive/MyDrive/Colab Notebooks/기계학습/train.txt', device)

## NN 모델 만들기 (마지막 layer는 0과 1 사이 값을 출력하도록 하기 위해서 signmoid 유지)
model = nn.Sequential(
    nn.Linear(2, 10, bias=True), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(10, 10, bias=True), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(10, 10, bias=True), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(10, 10, bias=True), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(10, 10, bias=True), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(10, 10, bias=True), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(10, 10, bias=True), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(10, 1, bias=True), nn.Sigmoid()).to(device)

# 이진분류 크로스엔트피 비용 함수
loss_func = torch.nn.BCELoss().to(device)

# 옵티마이저 함수 (역전파 알고리즘을 수행할 함수)
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)

# 학습 모드 셋팅
model.train()

DATA= [[0. 0. 0.]
 [0. 1. 1.]
 [1. 0. 1.]
 [1. 1. 0.]]
INPUT_FEATURES= [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
LABELS= [[0.]
 [1.]
 [1.]
 [0.]]


Sequential(
  (0): Linear(in_features=2, out_features=10, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=10, out_features=10, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.2, inplace=False)
  (6): Linear(in_features=10, out_features=10, bias=True)
  (7): ReLU()
  (8): Dropout(p=0.2, inplace=False)
  (9): Linear(in_features=10, out_features=10, bias=True)
  (10): ReLU()
  (11): Dropout(p=0.2, inplace=False)
  (12): Linear(in_features=10, out_features=10, bias=True)
  (13): ReLU()
  (14): Dropout(p=0.2, inplace=False)
  (15): Linear(in_features=10, out_features=10, bias=True)
  (16): ReLU()
  (17): Dropout(p=0.2, inplace=False)
  (18): Linear(in_features=10, out_features=10, bias=True)
  (19): ReLU()
  (20): Dropout(p=0.2, inplace=False)
  (21): Linear(in_features=10, out_features=1, bias=True)
  (22): Sigmoid()
)

In [ ]:
# 모델 학습
for epoch in range(2001):

  # 기울기 계산한 것들 초기화
  optimizer.zero_grad()

  # H(x) 계산: forward 연산
  hypothesis =model(input_features)

  # 비용 계산
  cost = loss_func(hypothesis, labels)

  # 역전파 수행
  cost.backward()
  optimizer.step()

  # 200 에폭마다 비용 출력
  if epoch % 200 == 0:
    print(epoch, cost.item())

0 0.6931410431861877
200 0.6930365562438965
400 0.692963719367981
600 0.6927772164344788
800 0.6919956803321838
1000 0.6730166673660278
1200 0.02142452262341976
1400 0.004531446378678083
1600 0.0021779348608106375
1800 0.0013584846165031195
2000 0.0009642943041399121


In [ ]:
# 평가 모드 셋팅 (학습 시에 적용했던 드랍 아웃 여부 등을 비적용)
model.eval()

# 역전파를 적용하지 않도록 context manager 설정
with torch.no_grad():
    hypothesis = model(input_features)
    logits = (hypothesis > 0.5).float()
    predicts = tensor2list(logits)
    golds = tensor2list(labels)
    print("PRED=",predicts)
    print("GOLD=",golds)
    print("Accuracy : {0:f}".format(accuracy_score(golds, predicts)))

PRED= [[0.0], [1.0], [1.0], [0.0]]
GOLD= [[0.0], [1.0], [1.0], [0.0]]
Accuracy : 1.000000


# 6. Residual Connection
> 가중치층을 우회하여 상위 층으로 직접 연결하는 것
> 추상화 정도(낮은 수준 추상화와 높은 수준 추상화)를 적절히 섞어주는 효과 -> 앙상블 효과를 통해 성능 개선